# Import libraries

In [1]:
import numpy as np
import cv2
import os
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV

# ELA function

In [2]:
def ela(imagePath, scale=10, quality=90):
    """
    Performs Error Level Analysis (ELA) on an image.

    Args:
        imagePath (str): The path to the image file.
        scale (int): The scale factor for resizing the image. Default is 10.

    Returns:
        elaImage (np.array): The ELA image.
    """
    # Load the image
    image = cv2.imread(imagePath)

    # Resize the image
    resizedImage = cv2.resize(image, (scale, scale))

    # Convert the image to JPEG
    cv2.imwrite("temp.jpg", resizedImage, [cv2.IMWRITE_JPEG_QUALITY, quality])

    # Load the JPEG image and compute the ELA image
    elaImage = cv2.imread("temp.jpg")
    elaImage = cv2.absdiff(resizedImage, elaImage)
    elaImage = cv2.cvtColor(elaImage, cv2.COLOR_BGR2GRAY)

    return elaImage

# RPA function

In [3]:
def rpa(elaImage, threshold=5):
    """
    Performs Residual Pixel Analysis (RPA) on an ELA image.

    Args:
        elaImage (np.array): The ELA image.

    Returns:
        tampered (bool): True if the image is tampered, False otherwise.
    """
    # Calculate the standard deviation of the ELA image
    stddev = np.std(elaImage)

    # Define the threshold for detecting tampered images
#     threshold = 5

    # If the standard deviation is greater than the threshold, the image is tampered
    tampered = stddev > threshold

    return tampered

# Detect fraud function

In [4]:
def detectFraud(imagePath, scale, quality, threshold):
    """
    Detects identity card fraud using ELA and RPA.

    Args:
        imagePath (str): The path to the image file.

    Returns:
        result (str): The result of the fraud detection, either "Genuine" or "Tampered".
    """
    
    # Perform ELA on the image
    elaImage = ela(imagePath, scale, quality)

    # Perform RPA on the ELA image
    tampered = rpa(elaImage, threshold)

    # Determine the result
    if tampered:
        result = 1
    else:
        result = 0

    return result

# Read dataset

In [5]:
authentic = '/kaggle/input/casia-20-image-tampering-detection-dataset/CASIA2/Au/'
tampered = '/kaggle/input/casia-20-image-tampering-detection-dataset/CASIA2/Tp/'

# Create X and Y

In [6]:
X = list()
Y = list()
for files in os.listdir(authentic):
    X.append(authentic+files)
    Y.append('Au')
for files in os.listdir(tampered):
    X.append(tampered+files)
    Y.append('Tp')

# Label encode Y

In [7]:
le = preprocessing.LabelEncoder()
y = le.fit_transform(Y)

# Train test split

In [8]:
xTrain, xTest, yTrain, yTest = train_test_split(X,y,train_size = 0.8, random_state=40,shuffle=True, stratify=y)

# Perform grid search on train data

In [9]:
tp,fp,fn = 0,0,0
for scale in [10,20,30]:
    for quality in [90]:
        for threshold in [5,7,9]:
            tp,fp,fn, score = 0,0,0,0
            for i in xTrain[:1000]:
                try:
                    cnt = detectFraud(i, scale, quality, threshold)
                    if cnt == 0 and yTrain[xTrain.index(i)] == 0:
                        tp += 1 
                    elif cnt == 0 and yTrain[xTrain.index(i)] == 1:
                        fp += 1
                    elif cnt == 1 and yTrain[xTrain.index(i)] == 0:
                        fn += 1
                        
                    if cnt == yTrain[xTrain.index(i)]:
                        score += 1
                except:
                    pass
            precision = tp/(tp+fp)
            recall = tp/(tp+fn)
            F1 = 2*precision*recall/(precision+recall)
            print(F1,score/1000, scale, quality, threshold)

0.6503937007874016 0.556 10 90 5
0.7134104833219878 0.579 10 90 7
0.7323395981853533 0.587 10 90 9
0.6857983811626196 0.573 20 90 5
0.7299077733860342 0.59 20 90 7
0.7452471482889734 0.598 20 90 9
0.7156726768377254 0.59 30 90 5
0.7397435897435897 0.594 30 90 7
0.7507846829880729 0.603 30 90 9


# Print results on test data

In [10]:
scale, quality, threshold = 30,90,9
tp,fp,fn, score = 0,0,0,0
for i in xTest[:200]:
    try:
        cnt = detectFraud(i, scale, quality, threshold)
        if cnt == 0 and yTest[xTest.index(i)] == 0:
            tp += 1 
        elif cnt == 0 and yTest[xTest.index(i)] == 1:
            fp += 1
        elif cnt == 1 and yTest[xTest.index(i)] == 0:
            fn += 1

        if cnt == yTest[xTest.index(i)]:
            score += 1
    except:
        pass
precision = tp/(tp+fp)
recall = tp/(tp+fn)
F1 = 2*precision*recall/(precision+recall)
print(F1,score/200, scale, quality, threshold)

0.7081967213114755 0.555 30 90 9
